# Lazypredict

In [2]:
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

In [3]:
df = pd.read_excel("../Data Final/Final_PM14-Toilet.xlsx")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 233 entries, 0 to 232
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Start_Time               233 non-null    datetime64[us]
 1   End_Time                 233 non-null    datetime64[us]
 2   Data_Count               233 non-null    int64         
 3   Mean_Creping             233 non-null    float64       
 4   Mean_Yankee Speed        233 non-null    float64       
 5   Mean_Pope Reel Speed     233 non-null    float64       
 6   Mean_Yankee Pressure     233 non-null    float64       
 7   Mean_Stock Flow          233 non-null    float64       
 8   Mean_Stock Consistency   233 non-null    float64       
 9   Mean_Flow Coating        233 non-null    float64       
 10  Mean_Flow Release        233 non-null    float64       
 11  Mean_Jet Wire Ratio      233 non-null    float64       
 12  Mean_Load KWH Refiner    233 non-null    float6

In [4]:

# Cleaning
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[df['GSM'].str.len() <= 4].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)

# Create Pseudo_Mass Feature
df['pseudo_mass'] = (df['Mean_Stock Flow'] * df['Mean_Stock Consistency'])/ df['Mean_Yankee Speed']

# Create Coating-Release Ratio Feature
df['coating_release_ratio'] = df['Mean_Flow Coating'] / df['Mean_Flow Release']

# X Variables
features = [
    'Mean_Yankee Pressure',
    'Mean_Creping',
    '% NBKP',
    'Mean_Load KWH Refiner',
    'Mean_Jet Wire Ratio',
    'GSM',
    'coating_release_ratio',
    'pseudo_mass'
]
X = df[features]

# Y Variables
y = df['MDT']

In [5]:
X.tail()

,Mean_Yankee Pressure,Mean_Creping,% NBKP,Mean_Load KWH Refiner,Mean_Jet Wire Ratio,GSM,coating_release_ratio,pseudo_mass
228,7.398770,28.001003,18.893424,223.481410,0.930000,12.5,0.786670,3.030610
229,7.397889,27.649638,18.893424,229.104467,0.930000,12.5,0.786669,3.065399
230,7.397871,27.200588,18.893424,232.056306,0.932419,12.5,0.735810,3.065657
231,7.398178,27.217770,18.893424,240.124594,0.940000,12.5,0.679804,3.061127
232,7.398265,28.121759,18.893424,233.135451,0.940000,12.5,0.695135,3.017183


In [6]:
y.head()

0    488
1    416
2    583
3    546
4    481
Name: MDT, dtype: int64

In [7]:
print("Min Pseudo Mass: ", X['pseudo_mass'].min())
print("Max Pseudo Mass: ", X['pseudo_mass'].max())

Min Pseudo Mass:  2.731759441479051
Max Pseudo Mass:  3.6728115600654707


In [7]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

In [9]:
# Hitung MAPE untuk tiap model
mape_scores = {}

for model_name, y_pred in predictions.items():
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mape_scores[model_name] = mape

# Tambahkan ke tabel hasil
models["MAPE"] = pd.Series(mape_scores)

# Urutkan (semakin kecil semakin baik)
models = models.sort_values(by="MAPE")

In [10]:
print(models)

                               Adjusted R-Squared   R-Squared         RMSE  \
Model                                                                        
HistGradientBoostingRegressor            0.364199    0.474773    58.687620   
ExtraTreesRegressor                      0.326047    0.443256    60.422761   
OrthogonalMatchingPursuitCV              0.277310    0.402995    62.569381   
LassoLarsCV                              0.274440    0.400624    62.693515   
LarsCV                                   0.274440    0.400624    62.693515   
LassoCV                                  0.274146    0.400382    62.706191   
GammaRegressor                           0.264664    0.392549    63.114438   
OrthogonalMatchingPursuit                0.260100    0.388778    63.310013   
ElasticNet                               0.257949    0.387001    63.401972   
LassoLars                                0.249534    0.380050    63.760453   
Lasso                                    0.249459    0.379988   

# Regressor

In [11]:
# Import Libraries
import numpy as np

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error)

# Model Dasar
model = ExtraTreesRegressor(random_state=42,n_jobs=1)

# Grid Search Hyper Parameter
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 8, 10, 15],
    'min_samples_split': [2, 4, 6, 10],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 0.5, 0.7],
    'bootstrap': [True, False]
}

# K-Fold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# GGrid Search CV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1,
    verbose=1
)

# Training + Tuning
grid_search.fit(X, y)

# Model Terbaik
best_model = grid_search.best_estimator_
print("Best Parameters:")
print(grid_search.best_params_)

# Prrediksi
y_pred = best_model.predict(X)

# Matrik Evaluasi
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
epsilon = 1e-8
mape = np.mean(np.abs((y - y_pred) / (y + epsilon))) * 100

# Hasil
print("\n===== HASIL MODEL TERBAIK =====")
print(f"R2    : {r2:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"MAE   : {mae:.4f}")
print(f"MAPE  : {mape:.2f}%")

Fitting 5 folds for each of 1152 candidates, totalling 5760 fits
Best Parameters:
{'bootstrap': True, 'max_depth': 15, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}

===== HASIL MODEL TERBAIK =====
R2    : 0.9268
RMSE  : 24.1190
MAE   : 18.4762
MAPE  : 3.64%


### Pickle Files

In [12]:
import joblib

In [13]:
joblib.dump(best_model, 'model_PM14-Toilet.pkl')

['model_PM14-Toilet.pkl']

In [14]:
features_pkl = X.columns.tolist()
#joblib.dump(features_pkl, 'features_PM14-Toilet.pkl')

### Features Importance

In [15]:
# Feature Importance
import pandas as pd

feat_imp = pd.DataFrame({
    "Feature": features, 
    "Importance": model.feature_importances_
})

# Urutkan dari terbesar
feat_imp = feat_imp.sort_values(by="Importance", ascending=False)


# Plot
import matplotlib.pyplot as plt
plt.figure()
plt.barh(feat_imp["Feature"], feat_imp["Importance"])
plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Extra Tree")

plt.tight_layout()
plt.show()

NotFittedError: This ExtraTreesRegressor instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

### SHAP Analysis

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
print(list(X_train.columns))
print(list(X_test.columns))

In [ ]:
print(X_train.isnull().sum())
print(X_test.isnull().sum())

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

In [ ]:
shap.plots.scatter(shap_values[:, "Mean_Load KWH Refiner"])

In [ ]:
shap.plots.scatter(shap_values[:, "pseudo_mass"])

In [ ]:
shap_values = explainer.shap_values(np.array(X_test))
shap.initjs()
i = 0  # index data

shap.force_plot(
    explainer.expected_value,
    shap_values[i],
    X_test.iloc[i]
)